# 🎯 Kill Detector — train your own gameplay highlight model

End-to-end notebook: **data → train → evaluate → export to ONNX → push to Hugging Face**.

The model is a small **image classifier** (`kill` vs `no_kill`) fine-tuned from a
pretrained vision backbone. It plugs straight into the Montage Clipper app's AI tier
and runs **in the browser via Transformers.js (WebGPU)** — accurate and reliable
(unlike the generic VLM we were using).

### Before you start
1. **Runtime → Change runtime type → GPU** (free T4 is plenty).
2. You'll need a free **Hugging Face account** and a **write token**
   (https://huggingface.co/settings/tokens) for the final push step.

> First time? Leave `DEMO_MODE = True` (next cells) to run the whole pipeline on a
> tiny public dataset in ~5 minutes and confirm everything works — *before* you spend
> time labeling real frames.


In [ ]:
# 1) Check the GPU
!nvidia-smi -L || echo "No GPU — set Runtime > Change runtime type > GPU"


In [ ]:
# 2) Install dependencies (Colab already has torch/torchvision)
!pip -q install "transformers>=4.44" "datasets>=2.20" "evaluate" "accelerate>=0.30" \
  "optimum[exporters]>=1.20" "onnx" "onnxruntime" "scikit-learn" "matplotlib" "huggingface_hub>=0.24"
print("done")


## Configuration

`DEMO_MODE = True` trains on the tiny public `beans` dataset just to prove the
pipeline end-to-end. Set it to `False` once you have your own labeled frames in
`DATA_DIR` (a folder containing two subfolders: `kill/` and `no_kill/`).


In [ ]:
# 3) Configuration  -- edit these
DEMO_MODE  = True   # True = tiny public dataset (proves the pipeline); False = your data

BASE_MODEL = "google/vit-base-patch16-224"   # robust default
# For a much smaller/faster *browser* model, try: "WinKawaks/vit-tiny-patch16-224"

DATA_DIR   = "/content/drive/MyDrive/montage_dataset"  # used only when DEMO_MODE = False
OUTPUT_DIR = "/content/kill-detector"

HF_REPO_ID = "your-username/montage-kill-detector"     # <-- change to YOUR HF username

EPOCHS        = 5
BATCH_SIZE    = 16
LEARNING_RATE = 5e-5
VAL_SPLIT     = 0.15
SEED          = 42

import torch
print("CUDA available:", torch.cuda.is_available())


## Data

**Demo:** loads `beans` automatically.

**Your data:** mount Google Drive (next cell) and point `DATA_DIR` at a folder shaped
like this — the folder *names* become the labels:

```
montage_dataset/
├── kill/        # frames where the player gets a kill / is in active combat
│   ├── 0001.jpg
│   └── ...
└── no_kill/     # everything else (walking, looting, menus, downtime)
    ├── 0001.jpg
    └── ...
```

See `training/data/README.md` in the repo for how to extract and label frames.


In [ ]:
# 4) (Only when DEMO_MODE = False) Mount Google Drive so your dataset persists
if not DEMO_MODE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# 5) Load + split the dataset
from collections import Counter
from datasets import load_dataset

if DEMO_MODE:
    raw = load_dataset("beans")["train"]
    if "labels" in raw.column_names:
        raw = raw.rename_column("labels", "label")
else:
    raw = load_dataset("imagefolder", data_dir=DATA_DIR)["train"]  # column is already 'label'

split = raw.train_test_split(test_size=VAL_SPLIT, seed=SEED, stratify_by_column="label")
train_ds, val_ds = split["train"], split["test"]

labels   = train_ds.features["label"].names
id2label = {i: n for i, n in enumerate(labels)}
label2id = {n: i for i, n in enumerate(labels)}

print("Classes:", labels)
print("Train counts:", Counter(train_ds["label"]))
print("Val   counts:", Counter(val_ds["label"]))


## Preprocessing & augmentation

We resize/normalize exactly the way the backbone expects, and add light augmentation
(random crop, flip, color jitter) so the model generalizes across maps and skins.


In [ ]:
# 6) Image processor + transforms
from transformers import AutoImageProcessor
from torchvision.transforms import (Compose, Normalize, ToTensor, Resize,
                                     CenterCrop, RandomResizedCrop,
                                     RandomHorizontalFlip, ColorJitter)

processor = AutoImageProcessor.from_pretrained(BASE_MODEL)

size = processor.size
crop = size.get("shortest_edge") or size.get("height") or 224
normalize = Normalize(mean=processor.image_mean, std=processor.image_std)

train_tf = Compose([RandomResizedCrop(crop, scale=(0.7, 1.0)),
                    RandomHorizontalFlip(), ColorJitter(0.2, 0.2, 0.2),
                    ToTensor(), normalize])
val_tf   = Compose([Resize(crop), CenterCrop(crop), ToTensor(), normalize])

def apply_train(batch):
    batch["pixel_values"] = [train_tf(img.convert("RGB")) for img in batch["image"]]
    return batch
def apply_val(batch):
    batch["pixel_values"] = [val_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

train_ds.set_transform(apply_train)
val_ds.set_transform(apply_val)
print("input size:", crop)


In [ ]:
# 7) Model
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,   # replaces the pretrained head with a fresh one
)


In [ ]:
# 8) Metrics + batching. Kills are rare, so we report precision/recall/F1, not just accuracy.
import numpy as np, torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, preds), "precision": p, "recall": r, "f1": f1}

def collate_fn(examples):
    return {
        "pixel_values": torch.stack([e["pixel_values"] for e in examples]),
        "labels": torch.tensor([e["label"] for e in examples]),
    }


In [ ]:
# 9) Class-weighted loss so the rare 'kill' class isn't ignored
import torch.nn as nn
from transformers import Trainer

counts = Counter(train_ds["label"])
total  = sum(counts.values())
weights = torch.tensor(
    [total / (len(labels) * counts[i]) for i in range(len(labels))], dtype=torch.float
)
print("class weights:", weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.functional.cross_entropy(
            outputs.logits, labels, weight=weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss


In [ ]:
# 10) Training configuration
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    remove_unused_columns=False,   # keep 'image' so set_transform can run
    report_to="none",
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collate_fn, compute_metrics=compute_metrics,
)


In [ ]:
# 11) Train  (a few minutes on a T4)
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Saved model to", OUTPUT_DIR)


## Evaluate

Look at **recall for the `kill` class** especially — that's "of all real kills, how
many did we catch?" The confusion matrix shows where it gets confused.


In [ ]:
# 12) Evaluation report + confusion matrix
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

pred   = trainer.predict(val_ds)
y_pred = np.argmax(pred.predictions, axis=1)
y_true = pred.label_ids
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix")
plt.tight_layout(); plt.show()


In [ ]:
# 13) Sanity check on one image
from transformers import pipeline
clf = pipeline("image-classification", model=OUTPUT_DIR,
               device=0 if torch.cuda.is_available() else -1)
print(clf(val_ds[0]["image"]))


## Export to ONNX (for the browser)

Transformers.js runs ONNX models. We export the fine-tuned classifier and also make a
quantized (int8) copy so the browser download is small.


In [ ]:
# 14) Export to ONNX + quantize
import os, shutil
ONNX_TMP = "/content/onnx_tmp"
!optimum-cli export onnx --model {OUTPUT_DIR} --task image-classification {ONNX_TMP}

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(f"{ONNX_TMP}/model.onnx", f"{ONNX_TMP}/model_quantized.onnx",
                 weight_type=QuantType.QInt8)
print(sorted(os.listdir(ONNX_TMP)))


In [ ]:
# 15) Arrange the repo so it works for BOTH Python (root) and Transformers.js (onnx/)
os.makedirs(f"{OUTPUT_DIR}/onnx", exist_ok=True)
shutil.copy(f"{ONNX_TMP}/model.onnx",           f"{OUTPUT_DIR}/onnx/model.onnx")
shutil.copy(f"{ONNX_TMP}/model_quantized.onnx", f"{OUTPUT_DIR}/onnx/model_quantized.onnx")
print("root:", sorted(os.listdir(OUTPUT_DIR)))
print("onnx:", sorted(os.listdir(f"{OUTPUT_DIR}/onnx")))


In [ ]:
# 16) Quick ONNX sanity check
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession(f"{OUTPUT_DIR}/onnx/model.onnx", providers=["CPUExecutionProvider"])
dummy = np.random.randn(1, 3, crop, crop).astype(np.float32)
out = sess.run(None, {sess.get_inputs()[0].name: dummy})
print("ONNX output shape:", out[0].shape, "(should be [1, %d])" % len(labels))


## Push to Hugging Face

You'll be prompted for a **write** token. After this, your model lives at
`https://huggingface.co/<HF_REPO_ID>` and the app can load it by id.


In [ ]:
# 17) Write a model card, then push everything
card = "\n".join([
    "---",
    "license: mit",
    "library_name: transformers",
    "pipeline_tag: image-classification",
    "tags:",
    "- image-classification",
    "- transformers.js",
    "- gameplay",
    "---",
    "",
    "# Montage Kill Detector",
    "",
    "Binary classifier (kill / no_kill) for gameplay highlight detection, fine-tuned",
    "from `" + BASE_MODEL + "`. Runs in the browser via Transformers.js (WebGPU).",
    "",
    "## Labels",
] + [f"- `{n}`" for n in labels] + [
    "",
    "## Browser usage",
    "```js",
    "import { pipeline } from '@huggingface/transformers';",
    f"const clf = await pipeline('image-classification', '{HF_REPO_ID}', {{ device: 'webgpu' }});",
    "const out = await clf(imageUrl);  // [{label, score}, ...]",
    "```",
])
with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(card)

from huggingface_hub import login, HfApi
login()  # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi()
api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO_ID, repo_type="model")
print("Pushed -> https://huggingface.co/" + HF_REPO_ID)


## Use it in the Montage Clipper app

1. Copy `training/deployment/scorer_classifier.js` over `app/static/scorer.js`
   (it uses the reliable `image-classification` pipeline instead of the VLM).
2. Run the app with your model id:
   ```
   docker run --rm -p 8080:8000 -e BYPASS_CREDITS=true \
     -e VLM_MODEL_ID="your-username/montage-kill-detector" \
     ghcr.io/zlen2024/montage-clipper:latest
   ```
3. Pick **AI mode** — every sampled frame is now scored by *your* kill detector.

See `training/deployment/README.md` for details.
